# Benchmark Hybrid RAG-Web System on General Domain

Pengujian ini bertujuan untuk membuktikan bahwa sistem dapat menangani *general questions* di luar *scope* Danau Toba dengan memanfaatkan integrasi Web Search (DuckDuckGo) + Gemini.
Kita akan menggunakan sampel data dari **SQuAD** dan **HotpotQA**.

In [ ]:
import os
import sys
import json
import time
import re
import string
from collections import Counter

# Tambahkan path src agar bisa mengimport backend kita
sys.path.append(os.path.abspath('../src'))
from hybrid_system import HybridRAGSystem

In [ ]:
# Inisialisasi Sistem
system = HybridRAGSystem()
print("Sistem berhasil diinisialisasi.")

In [ ]:
def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(str(s)))))

def exact_match_score(prediction, ground_truth):
    return (normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

In [ ]:
def evaluate_dataset(dataset_name, dataset_path, num_samples=20):
    print(f"\n{'='*50}")
    print(f"Evaluating on {dataset_name}")
    print(f"{'='*50}")
    
    if not os.path.exists(dataset_path):
        print(f"File tidak ditemukan: {dataset_path}")
        print("Pastikan Anda telah menjalankan 'downloads.sh'.")
        return
        
    with open(dataset_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # Parsing SQuAD format atau format HotpotQA dasar
    qas = []
    if 'data' in data:  # SQuAD format
        for article in data['data']:
            for paragraph in article['paragraphs']:
                for qa in paragraph['qas']:
                    if not qa.get('is_impossible', False) and len(qa['answers']) > 0:
                        qas.append({
                            'question': qa['question'],
                            'answers': [a['text'] for a in qa['answers']]
                        })
                    if len(qas) >= num_samples:
                        break
                if len(qas) >= num_samples:
                    break
            if len(qas) >= num_samples:
                break
    elif isinstance(data, list): # HotpotQA list format
        for item in data:
            qas.append({
                'question': item['question'],
                'answers': [item['answer']]
            })
            if len(qas) >= num_samples:
                break
                
    print(f"Loaded {len(qas)} samples. Mulai evaluasi...")
    
    total_em = 0
    total_f1 = 0
    
    for i, item in enumerate(qas):
        question = item['question']
        gold_answers = item['answers']
        
        # Query sistem
        response = system.query(question)
        pred_answer = response.get('response', '')
        
        # Karena prediksi sistem bisa jadi kalimat panjang, 
        # kita evaluasi apakah jawaban ground_truth ada di dalam jawaban model (soft match) 
        # atau pakai F1 Score standard.
        best_f1 = max([f1_score(pred_answer, a) for a in gold_answers])
        best_em = max([exact_match_score(pred_answer, a) for a in gold_answers])
        
        total_em += best_em
        total_f1 += best_f1
        
        print(f"Q: {question}")
        print(f"A (Gold): {gold_answers[0]}")
        print(f"A (Pred): {pred_answer[:100]}...")
        print(f"F1: {best_f1:.2f} | EM: {int(best_em)}\n")
        
        time.sleep(2) # Hindari rate limit API
        
    avg_em = total_em / len(qas)
    avg_f1 = total_f1 / len(qas)
    print(f"\n>>> RESULTS {dataset_name} <<<")
    print(f"Avg Exact Match (EM) : {avg_em:.4f}")
    print(f"Avg F1 Score         : {avg_f1:.4f}")

In [ ]:
# Pastikan sudah unzip datanya dari script downloads.sh sebelum menjalankan.
# evaluate_dataset("SQuAD", "../datasets/squad/train-v1.1.json", num_samples=10)
# evaluate_dataset("HotpotQA", "../datasets/hotpotqa/hotpot_train_v1.1.json", num_samples=10)